Importer les bibliothèques nécessaires.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt


Charger les images du dataset et vérifier leurs extensions (jpeg, jpg, bmp, png). Supprimer celles qui ne correspondent pas.

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import CLASS_NAMES, CLASS_TO_INDEX, SUPPORTED_EXTENSIONS, load_dataset



DATA_DIR = PROJECT_ROOT / "Data"
records = []

for label in CLASS_NAMES:
    class_dir = DATA_DIR / label
    for path in class_dir.iterdir():
        if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS:
            records.append({"path": path , "label": label})


images, labels, image_paths, failures = load_dataset(records, root=PROJECT_ROOT)

print("Images:", images.shape, images.dtype)
print("Labels:", labels.shape, labels.dtype)
print("Label mapping:", CLASS_TO_INDEX)
for label in CLASS_NAMES:
    print(f"{label}: {np.count_nonzero(labels == CLASS_TO_INDEX[label])} images")
print(f"Files that could not be loaded: {len(failures)}")
if failures:
    display(pd.DataFrame(failures))


ModuleNotFoundError: No module named 'src'

Find duplicate images using SHA-256
Matching hashes identify identical file contents, not merely similar-looking images. This section reports duplicates without removing them.

In [ ]:
from src.inspect_dataset import sha256_file

file_hashes = []
for path in image_paths:
    file_hashes.append(sha256_file(Path(path)))

file_data = pd.DataFrame({
    "Path": image_paths,
    "Class": [CLASS_NAMES[label] for label in labels],
    "SHA256": file_hashes,
})

# keep=False includes every file in each duplicate group.
duplicates = file_data[file_data["SHA256"].duplicated(keep=False)]
duplicates = duplicates.sort_values("SHA256")

print("distinct hashes:", duplicates["SHA256"].nunique())
print("all files that have the duplication thing:", len(duplicates))
print("Extra copies only without original :", len(duplicates) - duplicates["SHA256"].nunique())
display(duplicates)


In [ ]:
to_keep = ~file_data["SHA256"].duplicated(keep= "first")
images = images[to_keep.to_numpy()]
labels = labels[to_keep.to_numpy()]
image_paths =file_data.loc[to_keep, "Path"].tolist()



Afficher graphiquement le nombre d’images dans chaque classe.

In [ ]:
class_data = pd.DataFrame({"Class": labels})
plt.figure(figsize=(8, 5))
ax = sns.countplot(data=class_data, x="Class", order=class_data["Class"].value_counts().index)
for container in ax.containers:
    ax.bar_label(container)

plt.title("Number of images per class")
plt.xlabel("Class")
plt.ylabel("Count")
plt.show()

Show one sample image from each class.

In [ ]:
plt.figure(figsize=(12, 3))

for class_id, class_name in enumerate(CLASS_NAMES):
    index = np.where(labels == class_id)[0][0]

    plt.subplot(1, 4, class_id + 1)
    plt.imshow(images[index])
    plt.title(class_name)
    plt.axis("off")

plt.tight_layout()
plt.show()


Split the data: 80% for training and 20% for testing.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    images,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

print("Training images:", X_train.shape)
print("Test images:", X_test.shape)
print("Training labels:", y_train.shape)
print("Test labels:", y_test.shape)


Normaliser les valeurs des pixels dans la plage [0, 1].

In [ ]:
X_train = X_train.astype(np.float32) / 255.0
X_test = X_test.astype(np.float32) / 255.0


Définir l’architecture CNN (Conv2D + MaxPooling + Dropout + Dense) avec des paramètres optimaux.

In [ ]:
from src.model import create_arch

model = create_arch()

Choisir les meilleures fonctions d’activation pour les couches cachées et la couche de sortie.

jai choisi relu pour la couche cachee, et softmax pour la couche de sortie ( parce que jai mutli classification )